In [34]:
import pandas as pd
from pathlib import Path 

In [35]:
csv_path = Path(r"C:\Users\aynur\Downloads\archive (9)\cleanned.csv")
df = pd.read_csv(csv_path)
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,num
0,63,Male,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,0
1,67,Male,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,2
2,67,Male,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,1
3,37,Male,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,0
4,41,Female,typical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,0


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       918 non-null    int64  
 1   sex       918 non-null    object 
 2   cp        918 non-null    object 
 3   trestbps  918 non-null    float64
 4   chol      918 non-null    float64
 5   fbs       918 non-null    bool   
 6   restecg   918 non-null    object 
 7   thalch    918 non-null    float64
 8   exang     918 non-null    bool   
 9   oldpeak   918 non-null    float64
 10  num       918 non-null    int64  
dtypes: bool(2), float64(4), int64(2), object(3)
memory usage: 66.5+ KB


In [37]:
df['num'].value_counts()

num
0    410
1    265
2    108
3    107
4     28
Name: count, dtype: int64

In [38]:
df['num'] = df['num'].apply(lambda x: 0 if x == 0 else 1)

In [39]:
df['num']

0      0
1      1
2      1
3      0
4      0
      ..
913    1
914    0
915    1
916    0
917    1
Name: num, Length: 918, dtype: int64

In [40]:
df['num'].value_counts()

num
1    508
0    410
Name: count, dtype: int64

In [41]:
X = df.drop('num', axis = 1)
y = df['num']
y = df['num'].values.ravel()  

In [42]:
df.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
num         0
dtype: int64

In [43]:
num_col = X.select_dtypes(include = 'number').columns.to_list()
cat_col = X.select_dtypes(include = 'object').columns.to_list()

In [44]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([
    ('scaling', StandardScaler())
])
cat_pipeline = Pipeline([
    ('encoding', OneHotEncoder(handle_unknown='ignore'))
]
)
preprocessing = ColumnTransformer([
    ('CAT', cat_pipeline, cat_col),
    ('NUM', num_pipeline, num_col)
])

X_arr = preprocessing.fit_transform(X)
X_prepared = pd.DataFrame(X_arr, columns = preprocessing.get_feature_names_out()) 


In [45]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [46]:
y = pd.DataFrame(y)

In [50]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    full_pipeline_rf,
    X_train, y_train,
    scoring='accuracy',
    cv=5
)

print("CV Accuracy-ləri:", scores)
print("Ortalama CV Accuracy:", scores.mean())
print("Std:", scores.std())

CV Accuracy-ləri: [0.78911565 0.85714286 0.86394558 0.7414966  0.80136986]
Ortalama CV Accuracy: 0.8106141086571614
Std: 0.04546216061001153


In [51]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__max_depth': [3, 5, 10, None],
    'model__min_samples_leaf': [1, 2, 3, 5],
    'model__n_estimators': [100, 200]
}

grid_search = GridSearchCV(
    full_pipeline_rf,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Ən yaxşı parametrlər:", grid_search.best_params_)
print("Ən yaxşı CV Accuracy:", grid_search.best_score_)

best_rf = grid_search.best_estimator_
print("Test Accuracy:", best_rf.score(X_test, y_test))

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Ən yaxşı parametrlər: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
Ən yaxşı CV Accuracy: 0.8187680551672724
Test Accuracy: 0.7663043478260869


In [60]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

best_rf = Pipeline([
    ('cleaning', preprocessing),
    ('model', RandomForestClassifier(
        max_depth=2,
        min_samples_leaf=3,
        n_estimators=20,
        random_state=42
    ))
])

best_rf.fit(X_train, y_train)
print("Train Accuracy:", best_rf.score(X_train, y_train))
print("Test Accuracy:", best_rf.score(X_test, y_test))

Train Accuracy: 0.7901907356948229
Test Accuracy: 0.7934782608695652
